# ============================================================
# STEP 3B — DEEP TABULAR MODELS (non-sklearn APIs)
# Turkish Quishing Detection v10
# ============================================================
# WHY THIS STEP:
#   Step 3 covered classical ML (trees + linear). This step adds
#   attention-based deep learning for TABULAR features, which:
#     - learns feature interactions automatically
#     - provides NATIVE feature importance (attention masks)
#     - sets up the central thesis comparison:
#         engineered features + trees   (Step 3)
#         engineered features + deep    (Step 3B, this file)
#         raw URL + sequence models     (Step 7, CNN+BiLSTM)
#
# Models:
#   1. TabNet            — attention-based, native interpretability
#   2. FT-Transformer    — transformer for tabular (optional)
#
# Uses the SAME domain-aware 5-fold CV, SAME 3 variants, SAME
# metrics as Step 3, and APPENDS to step3_all_metrics.csv so the
# leaderboard is directly comparable to classical models.
#
# Requires GPU. Install:
#   !pip install pytorch-tabnet -q
#   (FT-Transformer optional: !pip install tab-transformer-pytorch -q)
# ============================================================

In [ ]:
import os, time, json, warnings
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, f1_score, recall_score, precision_score,
    matthews_corrcoef, balanced_accuracy_score, accuracy_score,
    average_precision_score)

warnings.filterwarnings("ignore")
SEED = 42

# ── PATHS ─────────────────────────────────────────────────────
DATA_DIR = (
    "/content/drive/MyDrive/TUMC_dataset"
)
INPUT_FILE  = os.path.join(DATA_DIR, "features_full_final.csv")
METRICS_CSV = os.path.join(DATA_DIR, "step3_all_metrics_final.csv")  # append here
TABNET_IMP  = os.path.join(DATA_DIR, "tabnet_attention_importance.csv")

# ── Detect deep libs ──────────────────────────────────────────
HAS_TABNET = HAS_FTT = HAS_TORCH = False
try:
    import torch
    HAS_TORCH = True
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"
try:
    from pytorch_tabnet.tab_model import TabNetClassifier
    HAS_TABNET = True
except ImportError:
    pass
try:
    from tab_transformer_pytorch import FTTransformer
    HAS_FTT = True
except ImportError:
    pass

print("=" * 70)
print("STEP 3B — DEEP TABULAR MODELS")
print("=" * 70)
print(f"  Torch: {HAS_TORCH}  Device: {DEVICE}")
print(f"  TabNet: {HAS_TABNET}  FT-Transformer: {HAS_FTT}")
if DEVICE == "cpu":
    print("  ⚠ No GPU — TabNet will be slow. Enable GPU runtime in Colab.")

# ════════════════════════════════════════════════════════════
# CONFIG — match Step 3
# ════════════════════════════════════════════════════════════
TASKS    = ["binary", "5class"]
VARIANTS = {
    "A_all":         [],
    "B_no_tr":       ["is_tr_domain"],
    "C_no_tr_https": ["is_tr_domain", "is_https"],
}
# For speed: deep models only on the honest variant by default.
# Expand to all variants once you confirm it runs.
RUN_VARIANTS = ["C_no_tr_https"]   # change to list(VARIANTS) for full grid

# ════════════════════════════════════════════════════════════
# METRICS (identical to Step 3)
# ════════════════════════════════════════════════════════════
def compute_metrics(y_true, y_pred, y_proba, is_binary):
    m = {}
    m["accuracy"]     = accuracy_score(y_true, y_pred)
    m["balanced_acc"] = balanced_accuracy_score(y_true, y_pred)
    m["mcc"]          = matthews_corrcoef(y_true, y_pred)
    if is_binary:
        m["recall"]    = recall_score(y_true, y_pred, zero_division=0)
        m["precision"] = precision_score(y_true, y_pred, zero_division=0)
        m["f1"]        = f1_score(y_true, y_pred, zero_division=0)
        if y_proba is not None:
            try:
                m["auc"]    = roc_auc_score(y_true, y_proba)
                m["pr_auc"] = average_precision_score(y_true, y_proba)
            except Exception:
                m["auc"] = m["pr_auc"] = np.nan
    else:
        m["recall_macro"]    = recall_score(y_true, y_pred, average="macro", zero_division=0)
        m["precision_macro"] = precision_score(y_true, y_pred, average="macro", zero_division=0)
        m["f1_macro"]        = f1_score(y_true, y_pred, average="macro", zero_division=0)
        m["f1_weighted"]     = f1_score(y_true, y_pred, average="weighted", zero_division=0)
        if y_proba is not None:
            try:
                m["auc_ovr"] = roc_auc_score(
                    y_true, y_proba, multi_class="ovr", average="macro")
            except Exception:
                m["auc_ovr"] = np.nan
    return m


# ════════════════════════════════════════════════════════════
# TABNET — train one fold
# ════════════════════════════════════════════════════════════
def train_tabnet(X_tr, y_tr, X_te, y_te, n_classes, feat_names):
    """Returns (y_pred, y_proba, feature_importance)."""
    # Impute + scale (TabNet benefits from scaling)
    imp = SimpleImputer(strategy="median")
    sc  = StandardScaler()
    X_tr_s = sc.fit_transform(imp.fit_transform(X_tr))
    X_te_s = sc.transform(imp.transform(X_te))

    # Carve a validation split from train (10%) for early stopping
    n_val  = max(1, int(0.1 * len(X_tr_s)))
    rng    = np.random.RandomState(SEED)
    perm   = rng.permutation(len(X_tr_s))
    val_idx, fit_idx = perm[:n_val], perm[n_val:]

    clf = TabNetClassifier(
        n_d=32, n_a=32, n_steps=5, gamma=1.5,
        lambda_sparse=1e-4,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=2e-2),
        mask_type="entmax",
        scheduler_params=dict(step_size=50, gamma=0.9),
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        seed=SEED, verbose=0, device_name=DEVICE,
    )
    clf.fit(
        X_tr_s[fit_idx], y_tr[fit_idx],
        eval_set=[(X_tr_s[val_idx], y_tr[val_idx])],
        eval_metric=["accuracy"],
        max_epochs=100, patience=15,
        batch_size=4096, virtual_batch_size=512,
    )
    y_pred  = clf.predict(X_te_s)
    proba   = clf.predict_proba(X_te_s)
    y_proba = proba[:, 1] if n_classes == 2 else proba
    feat_imp = clf.feature_importances_   # native attention importance
    return y_pred, y_proba, feat_imp


# ════════════════════════════════════════════════════════════
# MAIN
# ════════════════════════════════════════════════════════════
if not HAS_TABNET:
    print("\n⚠ pytorch-tabnet not installed.")
    print("  Run: !pip install pytorch-tabnet -q")
    raise SystemExit

print(f"\n[1] Loading features ...")
df = pd.read_csv(INPUT_FILE, low_memory=False)
META = {"url","source","tld","label","label_enc",
        "class_final","class_enc","fold","reg_domain"}
ALL_FEATURES = [c for c in df.columns if c not in META]
folds = sorted(df["fold"].unique())
print(f"    Rows: {len(df):,}  Features: {len(ALL_FEATURES)}  Folds: {folds}")

# Load existing Step 3 metrics to append
if os.path.exists(METRICS_CSV):
    existing = pd.read_csv(METRICS_CSV)
    print(f"    Loaded existing leaderboard: {len(existing)} rows")
else:
    existing = pd.DataFrame()

new_rows = []
tabnet_importances = []

for task in TASKS:
    target_col = "label_enc" if task == "binary" else "class_enc"
    n_classes  = df[target_col].nunique()
    is_binary  = (task == "binary")
    print(f"\n{'='*70}")
    print(f"TASK: {task}  ({n_classes} classes)")
    print(f"{'='*70}")

    for variant_name in RUN_VARIANTS:
        drop_cols = VARIANTS[variant_name]
        feats = [f for f in ALL_FEATURES if f not in drop_cols]
        print(f"\n  ── {variant_name}: {len(feats)} features ──")

        # ── TabNet ───────────────────────────────────────────
        t0 = time.time()
        fold_metrics = []
        fold_imps    = []
        try:
            for test_fold in folds:
                tr = df[df["fold"] != test_fold]
                te = df[df["fold"] == test_fold]
                X_tr, y_tr = tr[feats].values, tr[target_col].values.astype(int)
                X_te, y_te = te[feats].values, te[target_col].values.astype(int)

                y_pred, y_proba, feat_imp = train_tabnet(
                    X_tr, y_tr, X_te, y_te, n_classes, feats)
                fm = compute_metrics(y_te, y_pred, y_proba, is_binary)
                fold_metrics.append(fm)
                fold_imps.append(feat_imp)
                print(f"      fold {test_fold}: done")

            avg = {k: np.nanmean([fm[k] for fm in fold_metrics])
                   for k in fold_metrics[0]}
            std = {k: np.nanstd([fm[k] for fm in fold_metrics])
                   for k in fold_metrics[0]}
            elapsed = time.time() - t0

            row = {"task": task, "variant": variant_name,
                   "model": "TabNet", "n_features": len(feats),
                   "train_time_s": round(elapsed, 1)}
            for k, v in avg.items():
                row[k] = round(v, 4)
                row[f"{k}_std"] = round(std[k], 4)
            new_rows.append(row)

            key = "auc" if is_binary else "f1_macro"
            print(f"    TabNet: {key}={avg.get(key, 0):.4f}  "
                  f"MCC={avg['mcc']:.4f}  ({elapsed:.0f}s)")

            # Save mean attention importance
            mean_imp = np.mean(fold_imps, axis=0)
            imp_df = pd.DataFrame({
                "feature": feats,
                "tabnet_importance": mean_imp,
                "task": task, "variant": variant_name,
            }).sort_values("tabnet_importance", ascending=False)
            tabnet_importances.append(imp_df)
            print(f"    Top-5 TabNet attention features:")
            for _, r in imp_df.head(5).iterrows():
                print(f"      {r['feature']:<25s}: {r['tabnet_importance']:.4f}")

        except Exception as e:
            print(f"    TabNet FAILED: {type(e).__name__}: {e}")

# ════════════════════════════════════════════════════════════
# SAVE — append to unified leaderboard
# ════════════════════════════════════════════════════════════
if new_rows:
    new_df = pd.DataFrame(new_rows)
    combined = pd.concat([existing, new_df], ignore_index=True) \
               if len(existing) else new_df
    combined.to_csv(METRICS_CSV, index=False)
    print(f"\n[2] Appended {len(new_rows)} rows → {METRICS_CSV}")

if tabnet_importances:
    pd.concat(tabnet_importances, ignore_index=True).to_csv(
        TABNET_IMP, index=False)
    print(f"    TabNet attention importance → {TABNET_IMP}")

# ════════════════════════════════════════════════════════════
# UNIFIED LEADERBOARD — classical vs deep
# ════════════════════════════════════════════════════════════
print(f"\n{'='*70}")
print("UNIFIED LEADERBOARD (classical + deep)")
print(f"{'='*70}")
final = pd.read_csv(METRICS_CSV)
for task in TASKS:
    sub = final[final["task"] == task]
    sort_col = "auc" if (task == "binary" and "auc" in sub.columns) else "f1_macro"
    if sort_col in sub.columns and len(sub):
        print(f"\n  {task.upper()} — top 8 by {sort_col}:")
        cols = [c for c in ["model","variant",sort_col,"mcc"] if c in sub.columns]
        print(sub.nlargest(8, sort_col)[cols].to_string(index=False))

print(f"""
{'='*70}
STEP 3B COMPLETE
{'='*70}
  Deep tabular model: TabNet (attention-based)
  Native interpretability: {TABNET_IMP}

  Thesis comparison now available:
    Classical trees (Step 3) vs Deep tabular (Step 3B)
    → if trees win: report "tree ensembles remain superior
      on engineered URL features despite deep alternatives"
    → if TabNet wins: report attention mechanism advantage

  Cross-check: compare TabNet attention top features against
  SHAP top features (Step 4) — agreement validates explainability

  Next → Step 4: SHAP (TreeExplainer) on best classical model
{'='*70}
""")

STEP 3B — DEEP TABULAR MODELS
  Torch: True  Device: cuda
  TabNet: True  FT-Transformer: False

[1] Loading features ...
    Rows: 1,239,308  Features: 134  Folds: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
    Loaded existing leaderboard: 72 rows

TASK: binary  (2 classes)

  ── C_no_tr_https: 132 features ──

Early stopping occurred at epoch 51 with best_epoch = 36 and best_val_0_accuracy = 0.99898
      fold 0: done
Stop training because you reached max_epochs = 100 with best_epoch = 97 and best_val_0_accuracy = 0.99926
      fold 1: done

Early stopping occurred at epoch 45 with best_epoch = 30 and best_val_0_accuracy = 0.99909
      fold 2: done
Stop training because you reached max_epochs = 100 with best_epoch = 86 and best_val_0_accuracy = 0.99918
      fold 3: done

Early stopping occurred at epoch 88 with best_epoch = 73 and best_val_0_accuracy = 0.99927
      fold 4: done
    TabNet: auc=0.9998  MCC=0.9820  (7776s)
    Top-5 TabNet attention features:
